# Lab 3: Predicting Software Ticket Resolution Time
**Goal:** Compare a mean baseline, Linear Regression, Polynomial Regression (degrees 2 and 3), Ridge, and Lasso.

This notebook uses `lab_ticket_resolution.csv`. Run the cells from top to bottom. If prompted, upload the CSV file you received from your teacher.

**Important feature decision:** `ticket_id` is only an identifier and is excluded. `lines_changed` is measured during/after the fix, and `reopened` may only be known after the ticket has been resolved, so both could leak future information. For a prediction made when a ticket first arrives, this notebook uses only `complexity_score` and `team_experience_years`. If your teacher intends a different prediction time, explain and justify the feature choice.

In [ ]:
# 1. Import libraries and load the CSV
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files
uploaded = files.upload()  # Choose lab_ticket_resolution.csv
filename = next(iter(uploaded))
df = pd.read_csv(filename)

print("Dataset shape:", df.shape)
display(df.head())
print("\nData types:")
print(df.dtypes)
print("\nDescriptive statistics:")
display(df.describe(include="all"))
print("\nMissing values:")
print(df.isna().sum())

## 2. Select target and features
- **Target (`y`):** `resolution_hours`
- **Features (`X`):** `complexity_score`, `team_experience_years`
- Excluded `ticket_id` because it is just an identifier.
- Excluded `lines_changed` and `reopened` because they may not be available at the time we want to make the prediction; using them could cause data leakage.

In [ ]:
# 2. Define X and y, then create a reproducible 80/20 train/test split
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = df[["complexity_score", "team_experience_years"]]
y = df["resolution_hours"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

cv = KFold(n_splits=5, shuffle=True, random_state=42)

## 3. Evaluation helper and mean baseline

In [ ]:
results = []

def evaluate_model(name, model, Xtr=X_train, Xte=X_test, ytr=y_train, yte=y_test):
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(yte, pred),
        "RMSE": np.sqrt(mean_squared_error(yte, pred)),
        "R2": r2_score(yte, pred)
    })
    return model, pred

baseline, baseline_pred = evaluate_model(
    "Mean baseline", DummyRegressor(strategy="mean")
)
display(pd.DataFrame(results).round(3))

## 4. Linear Regression

In [ ]:
linear_model, linear_pred = evaluate_model(
    "Linear Regression", LinearRegression()
)
display(pd.DataFrame(results).round(3))

## 5. Polynomial Regression: compare degree 2 and 3 with cross-validation

In [ ]:
poly_cv_rows = []
poly_models = {}
for degree in [2, 3]:
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        StandardScaler(),
        LinearRegression()
    )
    scores = cross_validate(
        model, X_train, y_train, cv=cv,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
            "r2": "r2"
        }
    )
    poly_cv_rows.append({
        "Degree": degree,
        "CV MAE": (-scores["test_mae"]).mean(),
        "CV RMSE": (-scores["test_rmse"]).mean(),
        "CV R2": scores["test_r2"].mean()
    })
    fitted, pred = evaluate_model(f"Polynomial degree {degree}", model)
    poly_models[degree] = (fitted, pred)

print("Cross-validation results on training data:")
display(pd.DataFrame(poly_cv_rows).round(3))
print("Test-set results:")
display(pd.DataFrame(results).round(3))

## 6. Ridge and Lasso: choose alpha using cross-validation

In [ ]:
# StandardScaler is important for Ridge/Lasso because they penalize coefficients.
ridge_search = GridSearchCV(
    make_pipeline(StandardScaler(), Ridge()),
    {"ridge__alpha": [0.01, 0.1, 1, 10, 100]},
    cv=cv, scoring="neg_mean_absolute_error"
)
ridge_search.fit(X_train, y_train)
ridge_model, ridge_pred = evaluate_model("Ridge", ridge_search.best_estimator_)
print("Selected Ridge alpha:", ridge_search.best_params_["ridge__alpha"])

lasso_search = GridSearchCV(
    make_pipeline(StandardScaler(), Lasso(max_iter=100000)),
    {"lasso__alpha": [0.001, 0.01, 0.1, 1, 10]},
    cv=cv, scoring="neg_mean_absolute_error"
)
lasso_search.fit(X_train, y_train)
lasso_model, lasso_pred = evaluate_model("Lasso", lasso_search.best_estimator_)
print("Selected Lasso alpha:", lasso_search.best_params_["lasso__alpha"])

results_df = pd.DataFrame(results).drop_duplicates(subset=["Model"], keep="last")
display(results_df.round(3).sort_values("MAE"))

## 7. Plot observed vs predicted and residuals

In [ ]:
# Plot for Linear Regression and Polynomial degree 2
predictions_to_plot = {
    "Linear Regression": linear_pred,
    "Polynomial degree 2": poly_models[2][1]
}

for name, pred in predictions_to_plot.items():
    plt.figure(figsize=(6, 5))
    plt.scatter(y_test, pred, alpha=0.8)
    low = min(y_test.min(), np.min(pred))
    high = max(y_test.max(), np.max(pred))
    plt.plot([low, high], [low, high], linestyle="--")
    plt.xlabel("Observed resolution hours")
    plt.ylabel("Predicted resolution hours")
    plt.title(f"Observed vs Predicted: {name}")
    plt.tight_layout()
    plt.show()

    residuals = y_test - pred
    plt.figure(figsize=(6, 5))
    plt.scatter(pred, residuals, alpha=0.8)
    plt.axhline(0, linestyle="--")
    plt.xlabel("Predicted resolution hours")
    plt.ylabel("Residual (observed - predicted)")
    plt.title(f"Residual Plot: {name}")
    plt.tight_layout()
    plt.show()

## 8. Recommendation and analysis questions

Use the printed tables and plots to answer these questions in your own words:

1. **Why is the mean baseline useful?** It gives a simple reference point. A useful model should perform better than always predicting the training-set average.
2. **Which model has the lowest MAE and RMSE?** Read the test-set comparison table. Lower MAE and RMSE are better; RMSE penalizes large errors more strongly.
3. **Did polynomial features improve validation performance?** Compare the degree 2 and degree 3 cross-validation MAE/RMSE with the linear model. A more complex model is not automatically better.
4. **How did regularization affect performance and coefficients?** Ridge shrinks coefficients; Lasso can shrink some coefficients to zero. Compare cross-validation and test scores.
5. **Why scale Ridge/Lasso?** Regularization penalizes coefficient sizes, so scaling helps features measured in different units be treated comparably.
6. **Potential leakage:** `lines_changed` and `reopened` may be known only during or after the fix. They were excluded for an early-ticket prediction.
7. **Would random splitting always be appropriate?** Not necessarily. For time-based data, use a time-based split; for repeated customers or teams, consider group-based splitting to reduce leakage between train and test.
8. **Before deployment:** test on newer real tickets, check missing values and outliers, assess errors across complexity/team groups, confirm feature availability, monitor performance drift, and get stakeholder review.

### Recommendation template
Choose the model with the best test performance **provided it also has reasonable cross-validation performance**. State its MAE, RMSE, and R² from your table. Since this is synthetic data, do not claim the results will automatically generalize to a real organization.